In [1]:

import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scanpy as sc
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from model.dataset import BagsDataset, custom_collate_fn
from model.model import MIL, EarlyStopping

In [2]:

def load_all_genes(reference_gene_file):
    all_genes = pd.read_csv(reference_gene_file)
    return all_genes['Gene'].values.tolist()


Define your interested geneset, in this study we use all human/mouse genes as our reference geneset


In [3]:
#define your interested geneset, in this study we use all human/mouse genes as our reference geneset
all_genes = pd.read_csv('data/human_filtered.csv')
all_genes

,Gene
0,C1orf141
1,PKP1
2,HIVEP3
3,GLMN
4,SLC44A5
...,...
23177,EPCAM
23178,CEACAM21
23179,CEACAM6
23180,KRT8


In [4]:
all_genes = all_genes['Gene'].values.tolist()

Load the data

In [52]:
# Load dataset and create DataLoader(details data structure in data preparation section)
adata = sc.read_h5ad('/work/DPDS/s439765/data4spacer/spatial_transcriptomics/health_tissue_visiumhd/Visium_HD_FF_Human_Tonsil_binned_outputs/T_cell_nc.h5ad')
dataset = BagsDataset(
    adata,
    immune_cell='tcell',
    radius=50,
    max_instances=500,
    n_genes=500,
    resolution='high',
    k=2,  # Ensure 'k' matches the number of bags per batch
)

Immune cell: T
[1 2]
Tumor cells shape after filtering: (428038, 18085)
Selecting top 500 genes based on mean expression


/work/OSPH/s439765/envs/spatial_tcr/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Preprocessed data: (463236, 560)


Creating Bags with radius 50: 100%|████████████████████████| 463236/463236 [10:35<00:00, 728.64it/s]


Total batches created: 26588


If you want run the model for multiple data

In [56]:
adatas = pd.read_csv('data/all_data/tcell_cosmx.csv')
adatas #make sure you have same data structure as in sample.csv

,adata,radius,resolution
0,/work/DPDS/s439765/data4spacer/spatial_transcr...,50,high
1,/work/DPDS/s439765/data4spacer/spatial_transcr...,50,high
2,/work/DPDS/s439765/data4spacer/spatial_transcr...,50,high
3,/work/DPDS/s439765/data4spacer/spatial_transcr...,50,high


In [24]:
dataset = BagsDataset(
    'data/all_data/tcell_cosmx.csv',
    immune_cell='tcell',
    max_instances=500,
    n_genes=500,
    k=2,  # Ensure 'k' matches the number of bags per batch
)

Immune cell: T
Reading adata from /work/DPDS/s439765/data4spacer/spatial_transcriptomics/linghua_cosmx/GAME3D6/d6_sample1_Tcell.h5ad
[1 0 2]
Tumor cells shape after filtering: (30903, 20385)
Selecting top 500 genes based on mean expression


/work/OSPH/s439765/envs/spatial_tcr/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Processing: adata=d6_sample1_Tcell.h5ad, radius=100, resolution=high
Reading adata from /work/DPDS/s439765/data4spacer/spatial_transcriptomics/linghua_cosmx/GAME3D6/d6_sample2_Tcell.h5ad
[0 1 2]
Tumor cells shape after filtering: (31353, 20385)
Selecting top 500 genes based on mean expression


/work/OSPH/s439765/envs/spatial_tcr/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Processing: adata=d6_sample2_Tcell.h5ad, radius=100, resolution=high


Creating Bags with radius 100: 100%|██████████████████████| 158845/158845 [00:31<00:00, 4985.87it/s]

Total batches created: 4554


In [53]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"Using device: {device} ({torch.cuda.get_device_name(torch.cuda.current_device())})")
else:
    print(f"Using device: {device}")
print("=====================================")

Using device: cpu


In [54]:
model = MIL(all_genes).to(device)
criterion = nn.BCELoss().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.02)
early_stopping = EarlyStopping(patience=5, delta=0.001)

In [55]:
output_dir = 'negative_control/tonsil_top500'
os.makedirs(output_dir, exist_ok=True)

In [56]:
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)

In [57]:
best_val_loss = float('inf')
best_model_path = os.path.join(output_dir, 'best_model.pth')

# Save spacer scores before training
spacer_scores_before_training = model.immunogenicity.ig.clone().detach().cpu()
spacer_scores_before_training = [score.item() for score in spacer_scores_before_training]

In [58]:
def save_metrics(epoch, train_loss, val_loss, val_auroc, a, b, alpha, beta, output_dir):
    file_path = os.path.join(output_dir, 'training_metrics.csv')
    if not os.path.exists(file_path):
        # Create the CSV file with headers
        with open(file_path, 'w') as f:
            f.write('Epoch,Train Loss,Val Loss,Val AUROC,a,b,alpha,beta\n')
    
    # Append metrics for the current epoch
    with open(file_path, 'a') as f:
        f.write(f'{epoch},{train_loss},{val_loss},{val_auroc},{a},{b},{alpha},{beta}\n')

def save_spacer_scores(epoch, all_genes, spacer_scores_before_training, spacer_scores_after_training, output_dir):
    # Create a DataFrame with IG scores before and after the current epoch
    spacer_score_data = {
        'Gene': all_genes,
        'SPACER Score Before Training': spacer_scores_before_training,
        'SPACER Score After Training': spacer_scores_after_training,
    }
    df = pd.DataFrame(spacer_score_data)
    
    # Calculate the difference and add it as a new column
    df['Difference'] = df['SPACER Score After Training'] - df['SPACER Score Before Training']
    df = df.sort_values(by='Difference', ascending=False)

    # Save to a CSV file for each epoch
    output_path = os.path.join(output_dir, f'spacer_score_changes_epoch_{epoch+1}.csv')
    df.to_csv(output_path, index=False)

Training

In [59]:
num_epochs = 10
selection = 'positive' # Choose 'positive(induce)' or 'negative(repel)' based on your research focus

In [60]:

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
        
    # Lists to store outputs and labels for AUROC calculation
    all_outputs = []
    all_labels = []
        
    with tqdm(train_loader, unit="batch") as tepoch:
        for i, batch_data in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")
            optimizer.zero_grad()

            # Unpack the batch data
            distances_list, gene_expressions_list, labels_list, core_idxs_list, gene_names_list, cell_ids_list = batch_data
                
            # Move data to device and prepare labels
            distances_list = [distances.to(device) for distances in distances_list]
            gene_expressions_list = [gene_exp.to(device) for gene_exp in gene_expressions_list]
            labels = torch.stack(labels_list).float().to(device)
            current_genes_list = gene_names_list  # List of gene names for each bag

            # Forward pass
            outputs = model(distances_list, gene_expressions_list, current_genes_list)
                
            if outputs is None:
                 continue  # Skip this batch if the model returns None
                
            if outputs.shape[0] != labels.shape[0]:
                # Handle mismatch in batch sizes if necessary
                continue
                
            # Compute BCE loss
            if selection == 'negative':
                labels = 1 - labels
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
            running_loss += loss.item()
            tepoch.set_postfix(loss=loss.item())
                
            # Accumulate outputs and labels for AUROC calculation
            all_outputs.extend(outputs.detach().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    train_loss = running_loss / len(train_loader)
        
    # Compute Training AUROC
    try:
        epoch_auc = roc_auc_score(all_labels, all_outputs)
    except ValueError:
        epoch_auc = float('nan')  # Handle case where AUROC can't be computed
        
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {train_loss:.4f}, AUROC: {epoch_auc:.4f}')
        
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_all_outputs = []
    val_all_labels = []
    with torch.no_grad():
        with tqdm(val_loader, unit="batch") as vtepoch:
            for val_batch_data in vtepoch:
                # Unpack validation batch data
                val_distances_list, val_gene_expressions_list, val_labels_list, val_core_idxs_list, val_gene_names_list, val_cell_ids_list = val_batch_data
                    
                # Move data to device and prepare labels
                val_distances_list = [distances.to(device) for distances in val_distances_list]
                val_gene_expressions_list = [gene_exp.to(device) for gene_exp in val_gene_expressions_list]
                val_labels = torch.stack(val_labels_list).float().to(device)
                val_current_genes_list = val_gene_names_list  # List of gene names for each bag
                    
                # Forward pass
                val_outputs = model(val_distances_list, val_gene_expressions_list, val_current_genes_list)
                    
                if val_outputs is None:
                    continue  # Skip this batch if the model returns None
                    
                if val_outputs.shape[0] != val_labels.shape[0]:
                    # Handle mismatch in batch sizes if necessary
                    continue
                    
                # Compute BCE loss
                if selection == 'negative':
                    val_labels = 1 - val_labels
                loss = criterion(val_outputs, val_labels)
                val_loss += loss.item()
                vtepoch.set_postfix(val_loss=loss.item())
                    
                # Accumulate outputs and labels for AUROC calculation
                val_all_outputs.extend(val_outputs.detach().cpu().numpy())
                val_all_labels.extend(val_labels.cpu().numpy())
            
        val_loss /= len(val_loader)
            
            # Compute Validation AUROC
        try:
            val_epoch_auc = roc_auc_score(val_all_labels, val_all_outputs)
        except ValueError:
            val_epoch_auc = float('nan')  # Handle case where AUROC can't be computed
            
        print(f'Validation Loss: {val_loss:.4f}, Validation AUROC: {val_epoch_auc:.4f}')

    # Save the best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"Best model saved with validation loss {val_loss:.4f}")
            
    torch.save(model.state_dict(), os.path.join(output_dir, f'model_epoch_{epoch+1}.pth'))
        
    a = model.distance.a.clone().detach().cpu().numpy()
    b = model.gene_expression.b.clone().detach().cpu()
    alpha = model.alpha.clone().detach().cpu()
    beta = model.beta.clone().detach().cpu()
    # Save metrics
    save_metrics(epoch+1, train_loss, val_loss, val_epoch_auc,a,b,alpha,beta, output_dir)

    # Save IG scores after each epoch
    spacer_scores_after_training = model.immunogenicity.ig.clone().detach().cpu()
    spacer_scores_after_training = [score.item() for score in spacer_scores_after_training]
    save_spacer_scores(epoch, all_genes, spacer_scores_before_training, spacer_scores_after_training, output_dir)

Epoch 1/10: 100%|██████████| 23929/23929 [03:46<00:00, 105.78batch/s, loss=0.159] 


Epoch [1/10], Loss: 0.4760, AUROC: 0.8689


100%|██████████| 2659/2659 [00:13<00:00, 203.47batch/s, val_loss=0.167] 


Validation Loss: 0.4600, Validation AUROC: 0.8776
Best model saved with validation loss 0.4600


Epoch 2/10: 100%|██████████| 23929/23929 [03:53<00:00, 102.32batch/s, loss=0.463] 


...

Validation Loss: 0.4683, Validation AUROC: 0.8831


Epoch 10/10: 100%|██████████| 23929/23929 [03:55<00:00, 101.56batch/s, loss=0.323] 


Epoch [10/10], Loss: 0.4569, AUROC: 0.8823


100%|██████████| 2659/2659 [00:12<00:00, 211.65batch/s, val_loss=0.216] 


Validation Loss: 0.4705, Validation AUROC: 0.8866


In [ ]:
adata = sc.read_h5ad('/work/DPDS/s439765/data4spacer/spatial_transcriptomics/linghua_cosmx/GAME3D6/d6_sample1_Tcell.h5ad')
adata.X.shape #auroc 0.8

(56900, 20385)

In [ ]:
adata = sc.read_h5ad('/work/DPDS/s439765/data4spacer/spatial_transcriptomics/linghua_cosmx/GAME3D6/d6_sample2_Tcell.h5ad')
adata.X.shape #auroc 0.58

(158845, 20385)

In [ ]:
adata = sc.read_h5ad('/work/DPDS/s439765/data4spacer/spatial_transcriptomics/linghua_cosmx/GAME3D10/d10_sample1_Tcell.h5ad')
adata.X.shape #auroc 0.55

(59835, 20385)

In [ ]:
adata = sc.read_h5ad('/work/DPDS/s439765/data4spacer/spatial_transcriptomics/linghua_cosmx/GAME3D10/d10_sample2_Tcell.h5ad')
adata.X.shape # auroc 0.55

(186570, 20385)

In [ ]:
#overall 0.68